# 0825_peace_014_joint_type_threshold_optimization

007 클래스 가중치 타입별 XGBoost 모델은 그대로 유지하고, 5개 타입 임계값을 공동 최적화한다. Walk-forward에서 전체 Recall 99% 제약의 미래 안정성을 확인한 뒤 FP가 가장 적은 정책을 선택한다.


## 1. 설정, 경로 탐색과 실행 로그


In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_014_joint_type_threshold_optimization"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def compute_scale_pos_weight(y: pd.Series) -> float:
    positive = int(y.sum())
    negative = int(len(y) - positive)
    if positive == 0 or negative == 0:
        raise ValueError("scale_pos_weight는 양성과 음성이 모두 있는 Train에서만 계산할 수 있습니다.")
    return negative / positive


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 15:16:48,023 | INFO | experiment=0825_peace_014_joint_type_threshold_optimization


2026-08-25 15:16:48,024 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 15:16:48,024 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 15:16:48,024 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 15:16:48,025 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 15:16:48,025 | INFO | log_file=docs/peace/0825_peace_014_joint_type_threshold_optimization.log


log saved to: docs/peace/0825_peace_014_joint_type_threshold_optimization.log


## 2. 원본 데이터와 매핑 검증

첫 번째 익명 인덱스 열은 `record_id`로 이름만 바꾸며 원본 파일은 수정하지 않습니다. 중복 제거는 원인 확인 전 데이터 의미를 바꿀 수 있어 이번 베이스라인에서 수행하지 않습니다.


In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 15:16:52,387 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 유효 피처

각 전문가 모델은 공통 `meta_feat1~4`와 `mapping.json`에 명시된 해당 타입의 `inspection_feat`만 사용합니다. 타입 분리 후 상수인 `inspection_type`과 식별자·시간·타깃은 입력에서 제외합니다.


In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 15:16:52,397 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 시간순 Train/Validation/Test 분할

전체 행의 누적 비율에 가장 가까운 timestamp 그룹 끝을 경계로 사용합니다. 같은 timestamp 그룹은 서로 다른 구간에 들어가지 않습니다.

- 0~70%: Train
- 70~80%: Validation
- 80~100%: 최종 Test


In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "fixed_test_threshold": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test     False
threshold_selected_on_test    False
fixed_test_threshold            0.5
Name: evaluation_policy, dtype: object

2026-08-25 15:16:52,761 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 15:16:52,761 | INFO | test_policy model_selection=False threshold=0.50


## 5. Peace 실험과 동일한 평가 지표

PR-AUC, ROC-AUC, Accuracy, Precision, Recall, F1, TP/FN/FP/TN, False Call Reduction을 계산합니다. Threshold 0.5는 베이스라인 비교용이며 운영 임계값이 아닙니다.


In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


2026-08-25 15:16:52,784 | INFO | threshold_selector_unit_test=PASS


## 6. 3-Fold Expanding Walk-forward 검증

첫 70% 개발 구간 안에서 Train을 누적 확장합니다. 각 Fold의 Calibration에서 임계값을 선택하고, 그 임계값을 바로 다음 미래 Evaluation에 고정 적용합니다.

| Fold | Train | Calibration | Evaluation |
|---|---:|---:|---:|
| Fold 1 | 0~30% | 30~40% | 40~50% |
| Fold 2 | 0~40% | 40~50% | 50~60% |
| Fold 3 | 0~50% | 50~60% | 60~70% |

Calibration과 Evaluation은 모델 학습에 사용하지 않으며, Evaluation은 임계값 선택에도 사용하지 않습니다.

In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 15:16:53,450 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

In [7]:
def fit_type_experts_for_fold(train_frame, calibration_frame, evaluation_frame, fold_name):
    calibration_probability = pd.Series(
        np.nan, index=calibration_frame.index, dtype="float64"
    )
    evaluation_probability = pd.Series(
        np.nan, index=evaluation_frame.index, dtype="float64"
    )
    training_rows = []

    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type]
        type_calibration = calibration_frame.loc[
            calibration_frame[TYPE_COLUMN] == inspection_type
        ]
        type_evaluation = evaluation_frame.loc[
            evaluation_frame[TYPE_COLUMN] == inspection_type
        ]
        y_train = type_train[TARGET].astype("int8")
        scale_pos_weight = compute_scale_pos_weight(y_train)

        assert len(type_train) > 0
        assert len(type_calibration) > 0
        assert len(type_evaluation) > 0
        assert y_train.nunique() == 2
        assert type_calibration[TARGET].nunique() == 2

        logger.info(
            "walk_forward_fit_start fold=%s type=%d train_rows=%d train_positive=%d calibration_rows=%d calibration_positive=%d evaluation_rows=%d evaluation_positive=%d scale_pos_weight=%.6f",
            fold_name,
            inspection_type,
            len(type_train),
            int(y_train.sum()),
            len(type_calibration),
            int(type_calibration[TARGET].sum()),
            len(type_evaluation),
            int(type_evaluation[TARGET].sum()),
            scale_pos_weight,
        )

        preprocessor = make_preprocessor(feature_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        X_calibration = preprocessor.transform(type_calibration[feature_columns])
        X_evaluation = preprocessor.transform(type_evaluation[feature_columns])

        model = XGBClassifier(**XGB_PARAMS, scale_pos_weight=scale_pos_weight)
        model.fit(X_train, y_train, verbose=False)
        calibration_probability.loc[type_calibration.index] = model.predict_proba(
            X_calibration
        )[:, 1]
        evaluation_probability.loc[type_evaluation.index] = model.predict_proba(
            X_evaluation
        )[:, 1]

        training_rows.append(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "train_rows": len(type_train),
                "train_positive": int(y_train.sum()),
                "calibration_rows": len(type_calibration),
                "calibration_positive": int(type_calibration[TARGET].sum()),
                "evaluation_rows": len(type_evaluation),
                "evaluation_positive": int(type_evaluation[TARGET].sum()),
                "raw_features": len(feature_columns),
                "encoded_features": X_train.shape[1],
                "scale_pos_weight": scale_pos_weight,
            }
        )
        logger.info("walk_forward_fit_done fold=%s type=%d", fold_name, inspection_type)
        del preprocessor, model, X_train, X_calibration, X_evaluation
        gc.collect()

    assert calibration_probability.notna().all()
    assert evaluation_probability.notna().all()
    return calibration_probability, evaluation_probability, training_rows


walk_forward_threshold_rows = []
walk_forward_metric_rows = []
walk_forward_type_evaluation_rows = []
walk_forward_training_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    segments = walk_forward_segments[fold_name]
    calibration_frame = segments["calibration"]
    evaluation_frame = segments["evaluation"]
    calibration_probability, evaluation_probability, training_rows = (
        fit_type_experts_for_fold(
            segments["train"], calibration_frame, evaluation_frame, fold_name
        )
    )
    walk_forward_training_rows.extend(training_rows)

    global_selection = select_threshold(
        calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL
    )
    walk_forward_threshold_rows.append(
        {"fold": fold_name, "scope": "global", **global_selection}
    )

    thresholds_by_type_fold = {}
    type_evaluation_prediction = pd.Series(
        np.nan, index=evaluation_frame.index, dtype="float64"
    )
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[
            calibration_frame[TYPE_COLUMN] == inspection_type
        ]
        type_calibration_probability = calibration_probability.loc[
            type_calibration.index
        ]
        selection = select_threshold(
            type_calibration[TARGET],
            type_calibration_probability,
            min_recall=MIN_RECALL,
        )
        threshold = selection["threshold"]
        thresholds_by_type_fold[inspection_type] = threshold
        walk_forward_threshold_rows.append(
            {
                "fold": fold_name,
                "scope": f"type_{inspection_type}",
                **selection,
            }
        )

        type_evaluation = evaluation_frame.loc[
            evaluation_frame[TYPE_COLUMN] == inspection_type
        ]
        type_evaluation_probability = evaluation_probability.loc[type_evaluation.index]
        type_prediction = (type_evaluation_probability >= threshold).astype("int8")
        type_evaluation_prediction.loc[type_evaluation.index] = type_prediction
        type_metrics = evaluate_predictions(
            type_evaluation[TARGET], type_prediction, type_evaluation_probability
        )
        type_metrics.update(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "threshold": threshold,
            }
        )
        walk_forward_type_evaluation_rows.append(type_metrics)

    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(
            evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD
        ),
        "global_threshold": evaluate_probabilities(
            evaluation_frame[TARGET],
            evaluation_probability,
            global_selection["threshold"],
        ),
        "type_specific_thresholds": evaluate_predictions(
            evaluation_frame[TARGET],
            type_evaluation_prediction,
            evaluation_probability,
        ),
    }
    for strategy, metrics in strategy_metrics.items():
        walk_forward_metric_rows.append(
            {"fold": fold_name, "strategy": strategy, **metrics}
        )

    logger.info(
        "walk_forward_fold_done fold=%s global_threshold=%.8f metrics=%s",
        fold_name,
        global_selection["threshold"],
        strategy_metrics,
    )

walk_forward_threshold_summary = pd.DataFrame(walk_forward_threshold_rows).set_index(
    ["fold", "scope"]
)
walk_forward_evaluation_metrics = pd.DataFrame(walk_forward_metric_rows).set_index(
    ["fold", "strategy"]
)
walk_forward_type_evaluation = pd.DataFrame(
    walk_forward_type_evaluation_rows
).set_index(["fold", "inspection_type"])
walk_forward_training_summary = pd.DataFrame(walk_forward_training_rows).set_index(
    ["fold", "inspection_type"]
)


2026-08-25 15:16:53,470 | INFO | walk_forward_fit_start fold=fold_1 type=0 train_rows=28277 train_positive=32 calibration_rows=8408 calibration_positive=11 evaluation_rows=6496 evaluation_positive=50 scale_pos_weight=882.656250


2026-08-25 15:16:53,966 | INFO | walk_forward_fit_done fold=fold_1 type=0


2026-08-25 15:16:53,994 | INFO | walk_forward_fit_start fold=fold_1 type=1 train_rows=22698 train_positive=269 calibration_rows=3868 calibration_positive=20 evaluation_rows=2618 evaluation_positive=186 scale_pos_weight=83.379182


2026-08-25 15:16:54,413 | INFO | walk_forward_fit_done fold=fold_1 type=1


2026-08-25 15:16:54,447 | INFO | walk_forward_fit_start fold=fold_1 type=2 train_rows=42288 train_positive=408 calibration_rows=16448 calibration_positive=92 evaluation_rows=18964 evaluation_positive=49 scale_pos_weight=102.647059


...

2026-08-25 15:17:01,150 | INFO | walk_forward_fit_start fold=fold_3 type=3 train_rows=67320 train_positive=622 calibration_rows=20747 calibration_positive=23 evaluation_rows=12673 evaluation_positive=3 scale_pos_weight=107.231511


2026-08-25 15:17:02,185 | INFO | walk_forward_fit_done fold=fold_3 type=3


2026-08-25 15:17:02,213 | INFO | walk_forward_fit_start fold=fold_3 type=4 train_rows=2771 train_positive=10 calibration_rows=698 calibration_positive=3 evaluation_rows=344 evaluation_positive=0 scale_pos_weight=276.100000


2026-08-25 15:17:02,456 | INFO | walk_forward_fit_done fold=fold_3 type=4


2026-08-25 15:17:02,746 | INFO | walk_forward_fold_done fold=fold_3 global_threshold=0.00013531 metrics={'fixed_0.5': {'rows': 43853, 'positive_samples': 39, 'tn': 42048, 'fp': 1766, 'fn': 11, 'tp': 28, 'accuracy': 0.9594782569037467, 'precision': 0.01560758082497213, 'recall': 0.717948717948718, 'false_call_reduction': 0.9596932487332815, 'f1': 0.03055100927441353, 'roc_auc': 0.9012521463108034, 'pr_auc': 0.010186606048544513}, 'global_threshold': {'rows': 43853, 'positive_samples': 39, 'tn': 1689, 'fp': 42125, 'fn': 0, 'tp': 39, 'accuracy': 0.039404373703053386, 'precision': 0.0009249596812446637, 'recall': 1.0, 'false_call_reduction': 0.038549322134477565, 'f1': 0.0018482098429021634, 'roc_auc': 0.9012521463108034, 'pr_auc': 0.010186606048544513}, 'type_specific_thresholds': {'rows': 43853, 'positive_samples': 39, 'tn': 12692, 'fp': 31122, 'fn': 2, 'tp': 37, 'accuracy': 0.2902652042049575, 'precision': 0.0011874578773388106, 'recall': 0.9487179487179487, 'false_call_reduction': 0.28

## 7. Walk-forward 미래 Evaluation 결과

공통·타입별 임계값은 각 Fold의 Calibration에서만 선택됐습니다. 아래 지표는 임계값 선택에 사용하지 않은 바로 다음 미래 Evaluation 결과입니다.

In [8]:
display(
    walk_forward_threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]
    ]
)
display(
    walk_forward_evaluation_metrics[
        [
            "positive_samples",
            "pr_auc",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(
    walk_forward_type_evaluation[
        [
            "threshold",
            "positive_samples",
            "pr_auc",
            "recall",
            "false_call_reduction",
            "tp",
            "fn",
        ]
    ]
)

walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index()
    .groupby("strategy")
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"),
        total_fn=("fn", "sum"),
    )
)
display(walk_forward_strategy_summary)
display(walk_forward_training_summary)
logger.info(
    "walk_forward_strategy_summary=%s",
    walk_forward_strategy_summary.to_dict(orient="index"),
)


threshold  positive_samples    recall  false_call_reduction  \
fold   scope                                                                 
fold_1 global   0.000194               200  0.990000              0.128075   
       type_0   0.002021                11  1.000000              0.908062   
       type_1   0.002628                20  1.000000              0.839137   
       type_2   0.000056                92  1.000000              0.009843   
       type_3   0.000469                73  1.000000              0.297017   
       type_4   0.000287                 4  1.000000              0.848558   
fold_2 global   0.001389               326  0.990798              0.275655   
       type_0   0.002599                50  1.000000              0.578809   
       type_1   0.001389               186  0.994624              0.347862   
       type_2   0.004600                49  1.000000              0.402326   
       type_3   0.135821                39  1.000000              0.789140   
       type_4   0.000138                 2  1.000000              0.003096   
fold_3 global   0.000135               152  0.993421              0.096264   
       type_0   0.000056                14  1.000000              0.031212   
       type_1   0.001551                80  1.000000              0.150718   
       type_2   0.001215                32  1.000000              0.211446   
       type_3   0.001731                23  1.000000              0.730409   
       type_4   0.020426                 3  1.000000              0.834532   

                tp  fn  
fold   scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
fold   strategy                                                          
fold_1 fixed_0.5                              326  0.245426   0.077366   
       global_threshold                       326  0.245426   0.007921   
       type_specific_thresholds               326  0.245426   0.008807   
fold_2 fixed_0.5                              152  0.042780   0.048139   
       global_threshold                       152  0.042780   0.005446   
       type_specific_thresholds               152  0.042780   0.009416   
fold_3 fixed_0.5                               39  0.010187   0.015608   
       global_threshold                        39  0.010187   0.000925   
       type_specific_thresholds                39  0.010187   0.001187   

                                   recall  false_call_reduction        f1  \
fold   strategy                                                             
fold_1 fixed_0.5                 0.576687              0.948712  0.136430   
       global_threshold          0.996933              0.068788  0.015716   
       type_specific_thresholds  0.981595              0.176145  0.017458   
fold_2 fixed_0.5                 0.493421              0.966322  0.087719   
       global_threshold          0.953947              0.398683  0.010831   
       type_specific_thresholds  0.802632              0.708527  0.018613   
fold_3 fixed_0.5                 0.717949              0.959693  0.030551   
       global_threshold          1.000000              0.038549  0.001848   
       type_specific_thresholds  0.948718              0.289679  0.002372   

                                  tp   fn     fp     tn  
fold   strategy                                          
fold_1 fixed_0.5                 188  138   2242  41472  
       global_threshold          325    1  40707   3007  
       type_specific_thresholds  320    6  36014   7700  
fold_2 fixed_0.5                  75   77   1483  42552  
       global_threshold          145    7  26479  17556  
       type_specific_thresholds  122   30  12835  31200  
fold_3 fixed_0.5                  28   11   1766  42048  
       global_threshold           39    0  42125   1689  
       type_specific_thresholds   37    2  31122  12692

threshold  positive_samples    pr_auc    recall  \
fold   inspection_type                                                    
fold_1 0                 0.002021                50  0.522045  1.000000   
       1                 0.002628               186  0.205211  0.978495   
       2                 0.000056                49  0.498703  1.000000   
       3                 0.000469                39  0.663668  1.000000   
       4                 0.000287                 2  0.005039  0.000000   
fold_2 0                 0.002599                14  0.005137  0.500000   
       1                 0.001389                80  0.189496  1.000000   
       2                 0.004600                32  0.041709  0.937500   
       3                 0.135821                23  0.002302  0.086957   
       4                 0.000138                 3  0.054571  1.000000   
fold_3 0                 0.000056                 4  0.003983  1.000000   
       1                 0.001551                25  0.017646  1.000000   
       2                 0.001215                 7  0.070176  0.857143   
       3                 0.001731                 3  0.123702  0.666667   
       4                 0.020426                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
fold   inspection_type                                 
fold_1 0                            0.522805   50   0  
       1                            0.465049  182   4  
       2                            0.004811   49   0  
       3                            0.195217   39   0  
       4                            0.195046    0   2  
fold_2 0                            0.726786    7   7  
       1                            0.474611   80   0  
       2                            0.430016   30   2  
       3                            0.896786    2  21  
       4                            0.010072    3   0  
fold_3 0                            0.027018    4   0  
       1                            0.072836   25   0  
       2                            0.192387    6   1  
       3                            0.713891    2   1  
       4                            0.816860    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.099464,0.596019,0.493421,0,0.958243,0.948712,291,226
global_threshold,3,0.099464,0.983627,0.953947,2,0.168673,0.038549,509,8
type_specific_thresholds,3,0.099464,0.910982,0.802632,0,0.391450,0.176145,479,38


train_rows  train_positive  calibration_rows  \
fold   inspection_type                                                 
fold_1 0                     28277              32              8408   
       1                     22698             269              3868   
       2                     42288             408             16448   
       3                     37264             510             14419   
       4                      1610               4               836   
fold_2 0                     36685              43              6496   
       1                     26566             289              2618   
       2                     58736             500             18964   
       3                     51683             583             15637   
       4                      2446               8               325   
fold_3 0                     43181              93              8985   
       1                     29184             475              5023   
       2                     77700             549              8734   
       3                     67320             622             20747   
       4                      2771              10               698   

                        calibration_positive  evaluation_rows  \
fold   inspection_type                                          
fold_1 0                                  11             6496   
       1                                  20             2618   
       2                                  92            18964   
       3                                  73            15637   
       4                                   4              325   
fold_2 0                                  50             8985   
       1                                 186             5023   
       2                                  49             8734   
       3                                  39            20747   
       4                                   2              698   
fold_3 0                                  14            12107   
       1                                  80             4693   
       2                                  32            14036   
       3                                  23            12673   
       4                                   3              344   

                        evaluation_positive  raw_features  encoded_features  \
fold   inspection_type                                                        
fold_1 0                                 50            48                80   
       1                                186            56               106   
       2                                 49            69               114   
       3                                 39            69               107   
       4                                  2            25                47   
fold_2 0                                 14            48                82   
       1                                 80            56               110   
       2                                 32            69               114   
       3                                 23            69               107   
       4                                  3            25                47   
fold_3 0                                  4            48                84   
       1                                 25            56               111   
       2                                  7            69               115   
       3                                  3            69               108   
       4                                  0            25                50   

                        scale_pos_weight  
fold   inspection_type                    
fold_1 0                      882.656250  
       1                       83.379182  
       2                      102.647059  
       3                       72.066667  
       4                      401.500000  
fold_2 0                      852.139535  
     

2026-08-25 15:17:02,769 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.09946418478844642, 'mean_recall': 0.5960189623815714, 'min_recall': 0.4934210526315789, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9582425251628667, 'min_false_call_reduction': 0.9487120830855104, 'total_tp': 291, 'total_fn': 226}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.09946418478844642, 'mean_recall': 0.983626627919492, 'min_recall': 0.9539473684210527, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.16867340644364104, 'min_false_call_reduction': 0.038549322134477565, 'total_tp': 509, 'total_fn': 8}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.09946418478844642, 'mean_recall': 0.910981539896619, 'min_recall': 0.8026315789473685, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.3914504493174436, 'min_false_call_reduction': 0.17614494212380474, 'total_tp': 479, 'total_fn': 38}}


## 8. 최종 타입별 전문가 모델 5개 학습

각 타입에서 전처리기는 Train에만 `fit`합니다. 클래스 가중치는 각 타입의 현재 Train에서만 `scale_pos_weight = 음성 수 / 양성 수`로 계산하고, 시간 가중치·리샘플링은 적용하지 않습니다.


In [9]:
pooled_probability = pd.Series(np.nan, index=validation_df.index, dtype="float64")
models_by_type = {}
preprocessors_by_type = {}
type_metric_rows = []
training_rows = []

for inspection_type in inspection_types:
    feature_columns = feature_columns_by_type[inspection_type]
    type_train = train_df.loc[train_df[TYPE_COLUMN] == inspection_type]
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    y_train = type_train[TARGET].astype("int8")
    y_validation = type_validation[TARGET].astype("int8")
    scale_pos_weight = compute_scale_pos_weight(y_train)

    assert len(type_train) > 0 and len(type_validation) > 0
    assert y_train.nunique() == 2, f"type={inspection_type} Train에 두 클래스가 없습니다."
    logger.info(
        "model_fit_start type=%d train_rows=%d train_positive=%d valid_rows=%d valid_positive=%d raw_features=%d scale_pos_weight=%.6f",
        inspection_type,
        len(type_train),
        int(y_train.sum()),
        len(type_validation),
        int(y_validation.sum()),
        len(feature_columns),
        scale_pos_weight,
    )

    preprocessor = make_preprocessor(feature_columns)
    X_train = preprocessor.fit_transform(type_train[feature_columns])
    X_validation = preprocessor.transform(type_validation[feature_columns])
    assert np.isfinite(X_train.data if hasattr(X_train, "data") else X_train).all()
    assert np.isfinite(X_validation.data if hasattr(X_validation, "data") else X_validation).all()

    model = XGBClassifier(**XGB_PARAMS, scale_pos_weight=scale_pos_weight)
    model.fit(X_train, y_train, verbose=False)
    probability = model.predict_proba(X_validation)[:, 1]
    pooled_probability.loc[type_validation.index] = probability

    metrics = evaluate_probabilities(y_validation, probability)
    metrics["inspection_type"] = inspection_type
    type_metric_rows.append(metrics)
    training_rows.append(
        {
            "inspection_type": inspection_type,
            "train_rows": len(type_train),
            "train_positive": int(y_train.sum()),
            "validation_rows": len(type_validation),
            "validation_positive": int(y_validation.sum()),
            "raw_features": len(feature_columns),
            "encoded_features": X_train.shape[1],
            "trees": model.n_estimators,
            "scale_pos_weight": scale_pos_weight,
        }
    )
    models_by_type[inspection_type] = model
    preprocessors_by_type[inspection_type] = preprocessor
    logger.info(
        "model_fit_done type=%d pr_auc=%.6f recall=%.6f fcr=%.6f tp=%d fn=%d",
        inspection_type,
        metrics["pr_auc"],
        metrics["recall"],
        metrics["false_call_reduction"],
        metrics["tp"],
        metrics["fn"],
    )
    del X_train, X_validation, probability
    gc.collect()

assert pooled_probability.notna().all()
pooled_metrics = pd.Series(
    evaluate_probabilities(validation_df[TARGET], pooled_probability),
    name="type_expert_validation",
)
type_metrics = pd.DataFrame(type_metric_rows).set_index("inspection_type")
training_summary = pd.DataFrame(training_rows).set_index("inspection_type")
logger.info("pooled_validation_metrics=%s", pooled_metrics.to_dict())


2026-08-25 15:17:02,840 | INFO | model_fit_start type=0 train_rows=64273 train_positive=111 valid_rows=13289 valid_positive=12 raw_features=48 scale_pos_weight=578.036036


2026-08-25 15:17:03,732 | INFO | model_fit_done type=0 pr_auc=0.005600 recall=0.416667 fcr=0.951344 tp=5 fn=7


2026-08-25 15:17:03,766 | INFO | model_fit_start type=1 train_rows=38900 train_positive=580 valid_rows=6422 valid_positive=224 raw_features=56 scale_pos_weight=66.068966


2026-08-25 15:17:04,309 | INFO | model_fit_done type=1 pr_auc=0.693146 recall=0.937500 fcr=0.906583 tp=210 fn=14


2026-08-25 15:17:04,352 | INFO | model_fit_start type=2 train_rows=100470 train_positive=588 valid_rows=7161 valid_positive=27 raw_features=69 scale_pos_weight=169.867347


...

2026-08-25 15:17:05,638 | INFO | model_fit_start type=3 train_rows=100740 train_positive=648 valid_rows=16252 valid_positive=21 raw_features=69 scale_pos_weight=154.462963


2026-08-25 15:17:06,989 | INFO | model_fit_done type=3 pr_auc=0.065237 recall=0.619048 fcr=0.975048 tp=13 fn=8


2026-08-25 15:17:07,025 | INFO | model_fit_start type=4 train_rows=3813 train_positive=13 valid_rows=902 valid_positive=73 raw_features=25 scale_pos_weight=292.307692


2026-08-25 15:17:07,446 | INFO | model_fit_done type=4 pr_auc=0.070060 recall=0.013699 fcr=0.997587 tp=1 fn=72


2026-08-25 15:17:07,507 | INFO | pooled_validation_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 41920.0, 'fp': 1749.0, 'fn': 112.0, 'tp': 245.0, 'accuracy': 0.957729523463408, 'precision': 0.12286860581745236, 'recall': 0.6862745098039216, 'false_call_reduction': 0.9599487050310289, 'f1': 0.20842194810718842, 'roc_auc': 0.8672000527523289, 'pr_auc': 0.41127236814767276}


## 9. 최종 Validation 결과


In [10]:
count_columns = ["rows", "positive_samples", "tn", "fp", "fn", "tp"]
type_metrics[count_columns] = type_metrics[count_columns].astype("int64")
display(pooled_metrics)
display(
    type_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(training_summary)


rows                    44026.000000
positive_samples          357.000000
tn                      41920.000000
fp                       1749.000000
fn                        112.000000
tp                        245.000000
accuracy                    0.957730
precision                   0.122869
recall                      0.686275
false_call_reduction        0.959949
f1                          0.208422
roc_auc                     0.867200
pr_auc                      0.411272
Name: type_expert_validation, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,13289,12,0.005600,0.863542,0.950862,0.007680,0.416667,0.951344,0.015083,5,7,646,12631
1,6422,224,0.693146,0.958477,0.907661,0.266160,0.937500,0.906583,0.414610,210,14,579,5619
2,7161,27,0.478327,0.934092,0.982125,0.120301,0.592593,0.983600,0.200000,16,11,117,7017
3,16252,21,0.065237,0.930169,0.974588,0.031100,0.619048,0.975048,0.059226,13,8,405,15826
4,902,73,0.070060,0.318076,0.917960,0.333333,0.013699,0.997587,0.026316,1,72,2,827


,train_rows,train_positive,validation_rows,validation_positive,raw_features,encoded_features,trees,scale_pos_weight
inspection_type,,,,,,,,
0,64273,111,13289,12,48,88,400,578.036036
1,38900,580,6422,224,56,113,400,66.068966
2,100470,588,7161,27,69,117,400,169.867347
3,100740,648,16252,21,69,109,400,154.462963
4,3813,13,902,73,25,53,400,292.307692


## 10. 최종 Validation에서 공통·타입별 임계값 선택

Test를 사용하지 않고 Validation Recall 99% 이상을 만족하는 후보 중 False Call Reduction이 최대인 임계값을 선택합니다. 동률이면 Recall, 다시 동률이면 threshold가 높은 후보를 선택합니다.

In [11]:
global_threshold_selection = select_threshold(
    validation_df[TARGET], pooled_probability, min_recall=MIN_RECALL
)

thresholds_by_type = {}
type_threshold_rows = []
type_validation_prediction = pd.Series(np.nan, index=validation_df.index, dtype="float64")

for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_probability = pooled_probability.loc[type_validation.index]
    selection = select_threshold(
        type_validation[TARGET], type_probability, min_recall=MIN_RECALL
    )
    thresholds_by_type[inspection_type] = selection["threshold"]
    selection["inspection_type"] = inspection_type
    type_threshold_rows.append(selection)
    type_validation_prediction.loc[type_validation.index] = (
        type_probability >= selection["threshold"]
    ).astype("int8")

type_threshold_selection = pd.DataFrame(type_threshold_rows).set_index("inspection_type")
global_validation_metrics = pd.Series(
    evaluate_probabilities(
        validation_df[TARGET],
        pooled_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_specific_validation_metrics = pd.Series(
    evaluate_predictions(
        validation_df[TARGET],
        type_validation_prediction,
        pooled_probability,
    ),
    name="type_specific_thresholds",
)

validation_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": pooled_metrics,
        "global_threshold": global_validation_metrics,
        "type_specific_thresholds": type_specific_validation_metrics,
    }
).T

threshold_summary = pd.concat(
    [
        pd.DataFrame(
            [{"scope": "global", **global_threshold_selection}]
        ).set_index("scope"),
        type_threshold_selection.rename_axis("scope"),
    ],
    axis=0,
)

display(
    threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(
    validation_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
logger.info("global_threshold_selection=%s", global_threshold_selection)
logger.info("type_threshold_selection=%s", type_threshold_selection.to_dict(orient="index"))
logger.info("validation_strategy_metrics=%s", validation_strategy_metrics.to_dict(orient="index"))

,threshold,positive_samples,recall,false_call_reduction,tp,fn,fp,tn
scope,,,,,,,,
global,0.000189,357,0.991597,0.159587,354,3,36700,6969
0,0.000624,12,1.000000,0.557807,12,0,5871,7406
1,0.001798,224,0.991071,0.307357,222,2,4293,1905
2,0.003419,27,1.000000,0.368657,27,0,4504,2630
3,0.000544,21,1.000000,0.433060,21,0,9202,7029
4,0.000122,73,1.000000,0.000000,73,0,829,0


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.411272,0.122869,0.686275,0.959949,0.208422,245.0,112.0,1749.0,41920.0
global_threshold,0.411272,0.009554,0.991597,0.159587,0.018925,354.0,3.0,36700.0,6969.0
type_specific_thresholds,0.411272,0.014169,0.994398,0.434404,0.027941,355.0,2.0,24699.0,18970.0


2026-08-25 15:17:07,709 | INFO | global_threshold_selection={'threshold': 0.00018872492364607751, 'min_recall': 0.99, 'rows': 44026, 'positive_samples': 357, 'tn': 6969, 'fp': 36700, 'fn': 3, 'tp': 354, 'accuracy': 0.16633353018670785, 'precision': 0.009553624440006478, 'recall': 0.9915966386554622, 'false_call_reduction': 0.1595868923034647, 'f1': 0.018924915131913075, 'roc_auc': 0.8672000527523289, 'pr_auc': 0.41127236814767276}


2026-08-25 15:17:07,710 | INFO | type_threshold_selection={0: {'threshold': 0.0006238860660232604, 'min_recall': 0.99, 'rows': 13289, 'positive_samples': 12, 'tn': 7406, 'fp': 5871, 'fn': 0, 'tp': 12, 'accuracy': 0.5582060350665964, 'precision': 0.002039775624681285, 'recall': 1.0, 'false_call_reduction': 0.5578067334488213, 'f1': 0.004071246819338422, 'roc_auc': 0.8635422158620171, 'pr_auc': 0.005599591714426853}, 1: {'threshold': 0.0017983433790504932, 'min_recall': 0.99, 'rows': 6422, 'positive_samples': 224, 'tn': 1905, 'fp': 4293, 'fn': 2, 'tp': 222, 'accuracy': 0.3312052320149486, 'precision': 0.04916943521594684, 'recall': 0.9910714285714286, 'false_call_reduction': 0.3073572120038722, 'f1': 0.09369065203629458, 'roc_auc': 0.9584766687410685, 'pr_auc': 0.6931458622066441}, 2: {'threshold': 0.0034189000725746155, 'min_recall': 0.99, 'rows': 7161, 'positive_samples': 27, 'tn': 2630, 'fp': 4504, 'fn': 0, 'tp': 27, 'accuracy': 0.37103756458595166, 'precision': 0.005958949459280512, 

2026-08-25 15:17:07,710 | INFO | validation_strategy_metrics={'fixed_0.5': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 41920.0, 'fp': 1749.0, 'fn': 112.0, 'tp': 245.0, 'accuracy': 0.957729523463408, 'precision': 0.12286860581745236, 'recall': 0.6862745098039216, 'false_call_reduction': 0.9599487050310289, 'f1': 0.20842194810718842, 'roc_auc': 0.8672000527523289, 'pr_auc': 0.41127236814767276}, 'global_threshold': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 6969.0, 'fp': 36700.0, 'fn': 3.0, 'tp': 354.0, 'accuracy': 0.16633353018670785, 'precision': 0.009553624440006478, 'recall': 0.9915966386554622, 'false_call_reduction': 0.1595868923034647, 'f1': 0.018924915131913075, 'roc_auc': 0.8672000527523289, 'pr_auc': 0.41127236814767276}, 'type_specific_thresholds': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 18970.0, 'fp': 24699.0, 'fn': 2.0, 'tp': 355.0, 'accuracy': 0.4389451687639122, 'precision': 0.014169394108725154, 'recall': 0.9943977591036415, 'false_call_reduction

## 11. 전체 Recall 제약 기반 타입별 임계값 공동 최적화

각 타입을 독립적으로 최적화하지 않는다. 타입별 `(FN, FP)` Pareto 후보를 만든 뒤 동적 계획법으로 전체 FN 예산 안에서 총 FP가 최소인 임계값 조합을 정확히 선택한다. Calibration 목표 Recall은 99.0%·99.5%·100%만 비교하고, 선택에는 Walk-forward Evaluation만 사용한다.


In [12]:
from IPython.display import Markdown

SAFETY_TARGETS = [0.99, 0.995, 1.0]
OPERATING_RECALL = 0.99


def threshold_frontier_for_type(y_true, probability):
    """타입 하나의 threshold별 결과를 FN별 최소 FP Pareto 후보로 축약한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )
    total_positive = int((y_true == 1).sum())
    rows = [
        {
            "threshold": float(np.nextafter(sorted_probability.max(), np.inf)),
            "fn": total_positive,
            "fp": 0,
        }
    ]
    for position in group_ends:
        rows.append(
            {
                "threshold": float(sorted_probability[position]),
                "fn": int(total_positive - cumulative_tp[position]),
                "fp": int(cumulative_fp[position]),
            }
        )
    frontier = pd.DataFrame(rows)
    frontier = (
        frontier.sort_values(["fn", "fp", "threshold"], ascending=[True, True, False])
        .drop_duplicates("fn", keep="first")
        .sort_values("fn")
        .reset_index(drop=True)
    )
    return frontier


def predict_with_type_thresholds(frame, probability, thresholds):
    prediction = pd.Series(np.nan, index=frame.index, dtype="float64")
    for inspection_type, threshold in thresholds.items():
        mask = frame[TYPE_COLUMN] == inspection_type
        prediction.loc[mask] = (
            probability.loc[mask].to_numpy() >= threshold
        ).astype("int8")
    assert prediction.notna().all()
    return prediction.astype("int8")


def optimize_joint_type_thresholds(frame, probability, min_recall):
    """전체 Recall 제약 아래 총 FP가 최소인 타입별 threshold 조합을 정확히 선택한다."""
    total_positive = int(frame[TARGET].sum())
    fn_budget = int(np.floor((1.0 - min_recall) * total_positive + 1e-12))
    active_inspection_types = sorted(frame[TYPE_COLUMN].unique().tolist())
    frontiers = {}
    for inspection_type in active_inspection_types:
        mask = frame[TYPE_COLUMN] == inspection_type
        frontiers[inspection_type] = threshold_frontier_for_type(
            frame.loc[mask, TARGET], probability.loc[mask]
        )

    # state[사용 FN] = (총 FP, threshold dict). 같은 FN에서는 FP 최소만 유지한다.
    state = {0: (0, {})}
    for inspection_type in active_inspection_types:
        next_state = {}
        candidates = frontiers[inspection_type]
        candidates = candidates.loc[candidates["fn"] <= fn_budget]
        for used_fn, (used_fp, thresholds) in state.items():
            for candidate in candidates.itertuples(index=False):
                total_fn = used_fn + int(candidate.fn)
                if total_fn > fn_budget:
                    continue
                total_fp = used_fp + int(candidate.fp)
                current = next_state.get(total_fn)
                if current is None or total_fp < current[0]:
                    next_thresholds = dict(thresholds)
                    next_thresholds[inspection_type] = float(candidate.threshold)
                    next_state[total_fn] = (total_fp, next_thresholds)
        state = next_state
        if not state:
            raise RuntimeError(f"type={inspection_type}: 공동 threshold 최적화 상태가 비었습니다.")

    selected_fn, (selected_fp, thresholds) = min(
        state.items(), key=lambda item: (item[1][0], item[0])
    )
    prediction = predict_with_type_thresholds(frame, probability, thresholds)
    metrics = evaluate_predictions(frame[TARGET], prediction, probability)
    assert metrics["fn"] == selected_fn
    assert metrics["fp"] == selected_fp
    assert metrics["recall"] + 1e-12 >= min_recall
    return {
        "calibration_target": min_recall,
        "fn_budget": fn_budget,
        "thresholds": thresholds,
        **metrics,
    }


# 작은 합성 데이터에서 완전 탐색과 동적 계획법 결과가 같은지 확인한다.
_joint_test_frame = pd.DataFrame(
    {TYPE_COLUMN: [0, 0, 0, 1, 1, 1], TARGET: [1, 0, 0, 1, 0, 0]}
)
_joint_test_probability = pd.Series([0.9, 0.8, 0.1, 0.7, 0.6, 0.2])
_joint_test_result = optimize_joint_type_thresholds(
    _joint_test_frame, _joint_test_probability, min_recall=1.0
)
assert _joint_test_result["fn"] == 0
assert _joint_test_result["fp"] == 0


joint_walk_rows = []
joint_training_rows = []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    segments = walk_forward_segments[fold_name]
    calibration_probability_joint, evaluation_probability_joint, training_rows_joint = (
        fit_type_experts_for_fold(
            segments["train"], segments["calibration"], segments["evaluation"],
            f"{fold_name}_joint"
        )
    )
    joint_training_rows.extend(training_rows_joint)
    for safety_target in SAFETY_TARGETS:
        selection = optimize_joint_type_thresholds(
            segments["calibration"], calibration_probability_joint, safety_target
        )
        evaluation_prediction = predict_with_type_thresholds(
            segments["evaluation"], evaluation_probability_joint, selection["thresholds"]
        )
        evaluation_metrics = evaluate_predictions(
            segments["evaluation"][TARGET], evaluation_prediction, evaluation_probability_joint
        )
        joint_walk_rows.append(
            {
                "fold": fold_name,
                "calibration_target": safety_target,
                "calibration_fn_budget": selection["fn_budget"],
                "calibration_recall": selection["recall"],
                "calibration_fp": selection["fp"],
                "thresholds": selection["thresholds"],
                **{f"evaluation_{key}": value for key, value in evaluation_metrics.items()},
            }
        )
        logger.info(
            "joint_walk fold=%s target=%.4f calibration_fn=%d calibration_fp=%d evaluation_recall=%.6f evaluation_fp=%d evaluation_fcr=%.6f thresholds=%s",
            fold_name, safety_target, selection["fn"], selection["fp"],
            evaluation_metrics["recall"], evaluation_metrics["fp"],
            evaluation_metrics["false_call_reduction"], selection["thresholds"]
        )

joint_walk_metrics = pd.DataFrame(joint_walk_rows)
joint_walk_summary = (
    joint_walk_metrics.groupby("calibration_target", as_index=False)
    .agg(
        folds=("fold", "nunique"),
        recall_99_folds=("evaluation_recall", lambda values: int((values >= OPERATING_RECALL).sum())),
        min_recall=("evaluation_recall", "min"),
        mean_recall=("evaluation_recall", "mean"),
        total_fp=("evaluation_fp", "sum"),
        mean_false_call_reduction=("evaluation_false_call_reduction", "mean"),
        total_fn=("evaluation_fn", "sum"),
    )
    .sort_values(
        ["recall_99_folds", "min_recall", "total_fp", "mean_false_call_reduction", "calibration_target"],
        ascending=[False, False, True, False, False],
        kind="stable",
    )
    .reset_index(drop=True)
)
selected_safety_target = float(joint_walk_summary.iloc[0]["calibration_target"])
selected_walk_summary = joint_walk_summary.iloc[0]

display(
    joint_walk_metrics[
        ["fold", "calibration_target", "calibration_fn_budget", "calibration_recall",
         "evaluation_recall", "evaluation_fp", "evaluation_false_call_reduction",
         "evaluation_fn", "evaluation_tp"]
    ]
)
display(joint_walk_summary)
display(Markdown(f"**Walk-forward 선택 Calibration 목표 Recall: {selected_safety_target:.2%}**"))
logger.info(
    "joint_walk_selected target=%.4f summary=%s",
    selected_safety_target, selected_walk_summary.to_dict()
)


final_joint_selection = optimize_joint_type_thresholds(
    validation_df, pooled_probability, selected_safety_target
)
final_joint_thresholds = final_joint_selection["thresholds"]
joint_validation_prediction = predict_with_type_thresholds(
    validation_df, pooled_probability, final_joint_thresholds
)
joint_validation_metrics = pd.Series(
    evaluate_predictions(validation_df[TARGET], joint_validation_prediction, pooled_probability),
    name="joint_type_thresholds"
)
joint_threshold_table = pd.DataFrame(
    [{"inspection_type": key, "threshold": value} for key, value in final_joint_thresholds.items()]
).set_index("inspection_type")

display(joint_threshold_table)
display(joint_validation_metrics)
logger.info(
    "final_joint_validation target=%.4f fn_budget=%d thresholds=%s metrics=%s",
    selected_safety_target, final_joint_selection["fn_budget"], final_joint_thresholds,
    joint_validation_metrics.to_dict()
)


2026-08-25 15:17:07,769 | INFO | walk_forward_fit_start fold=fold_1_joint type=0 train_rows=28277 train_positive=32 calibration_rows=8408 calibration_positive=11 evaluation_rows=6496 evaluation_positive=50 scale_pos_weight=882.656250


2026-08-25 15:17:08,359 | INFO | walk_forward_fit_done fold=fold_1_joint type=0


2026-08-25 15:17:08,387 | INFO | walk_forward_fit_start fold=fold_1_joint type=1 train_rows=22698 train_positive=269 calibration_rows=3868 calibration_positive=20 evaluation_rows=2618 evaluation_positive=186 scale_pos_weight=83.379182


2026-08-25 15:17:09,624 | INFO | walk_forward_fit_done fold=fold_1_joint type=1


2026-08-25 15:17:09,656 | INFO | walk_forward_fit_start fold=fold_1_joint type=2 train_rows=42288 train_positive=408 calibration_rows=16448 calibration_positive=92 evaluation_rows=18964 evaluation_positive=49 scale_pos_weight=102.647059


...

**Walk-forward 선택 Calibration 목표 Recall: 100.00%**

2026-08-25 15:17:21,102 | INFO | joint_walk_selected target=1.0000 summary={'calibration_target': 1.0, 'folds': 3.0, 'recall_99_folds': 0.0, 'min_recall': 0.8026315789473685, 'mean_recall': 0.910981539896619, 'total_fp': 81629.0, 'mean_false_call_reduction': 0.37889982670664163, 'total_fn': 38.0}


,threshold
inspection_type,
0,0.000624
1,0.000984
2,0.003419
3,0.000544
4,0.000122


rows                    44026.000000
positive_samples          357.000000
tn                      18272.000000
fp                      25397.000000
fn                          0.000000
tp                        357.000000
accuracy                    0.423136
precision                   0.013862
recall                      1.000000
false_call_reduction        0.418420
f1                          0.027345
roc_auc                     0.867200
pr_auc                      0.411272
Name: joint_type_thresholds, dtype: float64

2026-08-25 15:17:21,227 | INFO | final_joint_validation target=1.0000 fn_budget=0 thresholds={0: 0.0006238860660232604, 1: 0.0009840558050200343, 2: 0.0034189000725746155, 3: 0.0005441136891022325, 4: 0.00012238798080943525} metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 18272.0, 'fp': 25397.0, 'fn': 0.0, 'tp': 357.0, 'accuracy': 0.4231363285331395, 'precision': 0.01386192436126427, 'recall': 1.0, 'false_call_reduction': 0.41842038975016604, 'f1': 0.02734479721190303, 'roc_auc': 0.8672000527523289, 'pr_auc': 0.41127236814767276}


## 12. 고정된 공동 임계값으로 최종 Test 추론

Walk-forward에서 안전 목표를 선택하고 Validation에서 5개 임계값을 확정한 뒤에만 Test를 평가한다.


In [13]:
test_probability = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_test_metric_rows = []

for inspection_type in inspection_types:
    feature_columns = feature_columns_by_type[inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    preprocessor = preprocessors_by_type[inspection_type]
    model = models_by_type[inspection_type]

    X_test = preprocessor.transform(type_test[feature_columns])
    probability = model.predict_proba(X_test)[:, 1]
    test_probability.loc[type_test.index] = probability

    metrics = evaluate_probabilities(type_test[TARGET], probability)
    metrics["inspection_type"] = inspection_type
    type_test_metric_rows.append(metrics)
    logger.info(
        "test_type_metrics type=%d pr_auc=%.6f recall=%.6f fcr=%.6f tp=%d fn=%d",
        inspection_type,
        metrics["pr_auc"],
        metrics["recall"],
        metrics["false_call_reduction"],
        metrics["tp"],
        metrics["fn"],
    )
    del X_test, probability
    gc.collect()

assert test_probability.notna().all()
test_metrics = pd.Series(
    evaluate_probabilities(test_df[TARGET], test_probability),
    name="type_expert_test",
)
type_test_metrics = pd.DataFrame(type_test_metric_rows).set_index("inspection_type")
type_test_metrics[count_columns] = type_test_metrics[count_columns].astype("int64")
fixed_test_metrics = test_metrics.copy()
fixed_test_metrics.name = "fixed_0.5"
global_test_metrics = pd.Series(
    evaluate_probabilities(
        test_df[TARGET],
        test_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_test_prediction = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_selected_test_rows = []
for inspection_type in inspection_types:
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    type_probability = test_probability.loc[type_test.index]
    threshold = thresholds_by_type[inspection_type]
    type_prediction = (type_probability >= threshold).astype("int8")
    type_test_prediction.loc[type_test.index] = type_prediction
    metrics = evaluate_predictions(type_test[TARGET], type_prediction, type_probability)
    metrics.update({"inspection_type": inspection_type, "threshold": threshold})
    type_selected_test_rows.append(metrics)

type_specific_test_metrics = pd.Series(
    evaluate_predictions(test_df[TARGET], type_test_prediction, test_probability),
    name="type_specific_thresholds",
)
test_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": fixed_test_metrics,
        "global_threshold": global_test_metrics,
        "type_specific_thresholds": type_specific_test_metrics,
    }
).T
type_selected_test_metrics = pd.DataFrame(type_selected_test_rows).set_index("inspection_type")

logger.info("pooled_test_metrics_fixed_0.5=%s", fixed_test_metrics.to_dict())
logger.info("test_strategy_metrics=%s", test_strategy_metrics.to_dict(orient="index"))

display(
    test_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
display(
    type_selected_test_metrics[
        ["threshold", "positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(fixed_test_metrics)
display(
    type_test_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)

2026-08-25 15:17:21,304 | INFO | test_type_metrics type=0 pr_auc=0.032155 recall=0.333333 fcr=0.890495 tp=65 fn=130


2026-08-25 15:17:21,363 | INFO | test_type_metrics type=1 pr_auc=0.434122 recall=0.812661 fcr=0.795457 tp=629 fn=145


2026-08-25 15:17:21,443 | INFO | test_type_metrics type=2 pr_auc=0.604051 recall=0.641587 fcr=0.971280 tp=469 fn=262


2026-08-25 15:17:21,604 | INFO | test_type_metrics type=3 pr_auc=0.386073 recall=0.601307 fcr=0.989658 tp=368 fn=244


2026-08-25 15:17:21,641 | INFO | test_type_metrics type=4 pr_auc=0.017805 recall=0.000000 fcr=0.948252 tp=0 fn=13


...

2026-08-25 15:17:22,039 | INFO | test_strategy_metrics={'fixed_0.5': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 80285.0, 'fp': 5442.0, 'fn': 794.0, 'tp': 1531.0, 'accuracy': 0.9291782128742107, 'precision': 0.2195611644916105, 'recall': 0.6584946236559139, 'false_call_reduction': 0.9365194162865842, 'f1': 0.3293181329318133, 'roc_auc': 0.8714556448320381, 'pr_auc': 0.31860498449750563}, 'global_threshold': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 8617.0, 'fp': 77110.0, 'fn': 35.0, 'tp': 2290.0, 'accuracy': 0.12386998591741244, 'precision': 0.028841309823677583, 'recall': 0.9849462365591398, 'false_call_reduction': 0.1005167566810923, 'f1': 0.056041602936677884, 'roc_auc': 0.8714556448320381, 'pr_auc': 0.31860498449750563}, 'type_specific_thresholds': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 27477.0, 'fp': 58250.0, 'fn': 141.0, 'tp': 2184.0, 'accuracy': 0.3368577658656249, 'precision': 0.036138597478240726, 'recall': 0.9393548387096774, 'false_call_reducti

,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.318605,0.219561,0.658495,0.936519,0.329318,1531.0,794.0,5442.0,80285.0
global_threshold,0.318605,0.028841,0.984946,0.100517,0.056042,2290.0,35.0,77110.0,8617.0
type_specific_thresholds,0.318605,0.036139,0.939355,0.320517,0.069600,2184.0,141.0,58250.0,27477.0


,threshold,positive_samples,pr_auc,precision,recall,false_call_reduction,tp,fn,fp,tn
inspection_type,,,,,,,,,,
0,0.000624,195,0.032155,0.015168,0.789744,0.481810,154,41,9999,9297
1,0.001798,774,0.434122,0.076637,0.985788,0.205926,763,11,9193,2384
2,0.003419,731,0.604051,0.047188,0.934337,0.303907,683,48,13791,6021
3,0.000544,612,0.386073,0.022728,0.933007,0.284761,571,41,24552,9775
4,0.000122,13,0.017805,0.017857,1.000000,0.000000,13,0,715,0


rows                    88052.000000
positive_samples         2325.000000
tn                      80285.000000
fp                       5442.000000
fn                        794.000000
tp                       1531.000000
accuracy                    0.929178
precision                   0.219561
recall                      0.658495
false_call_reduction        0.936519
f1                          0.329318
roc_auc                     0.871456
pr_auc                      0.318605
Name: fixed_0.5, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,19491,195,0.032155,0.702975,0.884921,0.029844,0.333333,0.890495,0.054783,65,130,2113,17183
1,12351,774,0.434122,0.872305,0.796535,0.209877,0.812661,0.795457,0.333599,629,145,2368,9209
2,20543,731,0.604051,0.896528,0.959548,0.451830,0.641587,0.971280,0.530243,469,262,569,19243
3,34939,612,0.386073,0.863252,0.982856,0.508990,0.601307,0.989658,0.551311,368,244,355,33972
4,728,13,0.017805,0.413771,0.931319,0.000000,0.000000,0.948252,0.000000,0,13,37,678


## 13. 공동 임계값 Test 결과, 문서화와 무결성 검증


In [14]:
REPORT_PATH = REPO_ROOT / "docs" / "experiments" / f"{EXPERIMENT_ID}.md"

joint_test_prediction = predict_with_type_thresholds(
    test_df, test_probability, final_joint_thresholds
)
joint_test_metrics = pd.Series(
    evaluate_predictions(test_df[TARGET], joint_test_prediction, test_probability),
    name="joint_type_thresholds"
)

comparison_validation = pd.DataFrame(
    {
        "007_global_threshold": global_validation_metrics,
        "007_independent_type_thresholds": type_specific_validation_metrics,
        "014_joint_type_thresholds": joint_validation_metrics,
    }
).T
comparison_test = pd.DataFrame(
    {
        "007_global_threshold": global_test_metrics,
        "007_independent_type_thresholds": type_specific_test_metrics,
        "014_joint_type_thresholds": joint_test_metrics,
    }
).T

display(Markdown("### Validation 비교"))
display(comparison_validation[["recall", "false_call_reduction", "tp", "fn", "fp", "tn", "pr_auc"]])
display(Markdown("### Test 비교"))
display(comparison_test[["recall", "false_call_reduction", "tp", "fn", "fp", "tn", "pr_auc"]])

joint_success = bool(
    joint_test_metrics["recall"] >= OPERATING_RECALL
    and joint_test_metrics["fp"] < global_test_metrics["fp"]
)


def percent(value):
    return f"{value * 100:.2f}%"


report_lines = [
    f"# {EXPERIMENT_ID}",
    "",
    "## 연결된 노트북",
    "",
    f"`notebooks/{EXPERIMENT_ID}.ipynb`",
    "",
    "## 상태",
    "",
    "완료",
    "",
    "## 목적",
    "",
    "007 클래스 가중치 타입별 XGBoost 모델을 유지하면서, 전체 Recall 99% 제약 아래 5개 타입 임계값을 공동 최적화해 총 FP를 줄이고 False Call Reduction을 높일 수 있는지 확인한다.",
    "",
    "## 007 대비 주요 변경사항",
    "",
    "- 007은 각 타입이 개별적으로 Recall 99%를 만족하도록 임계값을 선택했다.",
    "- 014는 전체 positive의 FN 예산을 타입 사이에 공동 배분하고, 그 예산 안에서 총 FP가 최소인 임계값 조합을 동적 계획법으로 정확히 선택했다.",
    "- Calibration 목표 Recall 99.0%·99.5%·100% 중 하나를 Walk-forward Evaluation 결과만으로 선택했다.",
    "- Test는 안전 목표와 최종 임계값 5개를 고정한 뒤 한 번만 평가했다.",
    "",
    "## Walk-forward 안전 목표 비교",
    "",
    joint_walk_summary.to_markdown(index=False),
    "",
    f"선택된 Calibration 목표 Recall은 **{selected_safety_target:.2%}**다.",
    "",
    "## 최종 타입별 임계값",
    "",
    joint_threshold_table.reset_index().to_markdown(index=False),
    "",
    "## Validation 비교",
    "",
    comparison_validation[["recall", "false_call_reduction", "tp", "fn", "fp", "tn", "pr_auc"]].reset_index(names="strategy").to_markdown(index=False),
    "",
    "## 최종 Test 비교",
    "",
    comparison_test[["recall", "false_call_reduction", "tp", "fn", "fp", "tn", "pr_auc"]].reset_index(names="strategy").to_markdown(index=False),
    "",
    "## 결론",
    "",
    f"- 공동 임계값의 Test Recall은 {percent(joint_test_metrics['recall'])}, FP는 {int(joint_test_metrics['fp']):,}, FCR은 {percent(joint_test_metrics['false_call_reduction'])}다.",
    f"- 007 공통 임계값 대비 FP 변화는 {int(joint_test_metrics['fp'] - global_test_metrics['fp']):+,}개, FCR 변화는 {(joint_test_metrics['false_call_reduction'] - global_test_metrics['false_call_reduction']) * 100:+.2f}%p다.",
    f"- 운영 성공 조건(Test Recall 99% 이상이면서 기존 공통 임계값보다 FP 감소)은 **{'충족' if joint_success else '미충족'}**이다.",
    "- Test는 재사용된 최종 확인 구간이며 안전 목표와 임계값 선택에는 사용하지 않았다.",
    "",
    "## 저장 모델",
    "",
    "해당 없음. 기존 007 모델 구조의 threshold 정책만 비교했다.",
    "",
    "## 실행 로그",
    "",
    f"`docs/peace/{EXPERIMENT_ID}.log`",
]
REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE
assert joint_test_prediction.notna().all()
assert set(final_joint_thresholds) == set(inspection_types)

verification = pd.Series(
    {
        "dataset_sha256_unchanged": True,
        "mapping_sha256_unchanged": True,
        "selected_safety_target": selected_safety_target,
        "walk_forward_recall_99_folds": int(selected_walk_summary["recall_99_folds"]),
        "final_type_threshold_count": len(final_joint_thresholds),
        "test_evaluated_after_selection": True,
        "success_condition_met": joint_success,
        "report_path": str(REPORT_PATH.relative_to(REPO_ROOT)),
        "log_path": str(LOG_PATH.relative_to(REPO_ROOT)),
    },
    name="verification",
)
display(verification)
logger.info("joint_test_metrics=%s", joint_test_metrics.to_dict())
logger.info("source_integrity=PASS")
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()


### Validation 비교

,recall,false_call_reduction,tp,fn,fp,tn,pr_auc
007_global_threshold,0.991597,0.159587,354.0,3.0,36700.0,6969.0,0.411272
007_independent_type_thresholds,0.994398,0.434404,355.0,2.0,24699.0,18970.0,0.411272
014_joint_type_thresholds,1.000000,0.418420,357.0,0.0,25397.0,18272.0,0.411272


### Test 비교

,recall,false_call_reduction,tp,fn,fp,tn,pr_auc
007_global_threshold,0.984946,0.100517,2290.0,35.0,77110.0,8617.0,0.318605
007_independent_type_thresholds,0.939355,0.320517,2184.0,141.0,58250.0,27477.0,0.318605
014_joint_type_thresholds,0.940645,0.308678,2187.0,138.0,59265.0,26462.0,0.318605


dataset_sha256_unchanged                                                       True
mapping_sha256_unchanged                                                       True
selected_safety_target                                                          1.0
walk_forward_recall_99_folds                                                      0
final_type_threshold_count                                                        5
test_evaluated_after_selection                                                 True
success_condition_met                                                         False
report_path                       docs/experiments/0825_peace_014_joint_type_thr...
log_path                          docs/peace/0825_peace_014_joint_type_threshold...
Name: verification, dtype: object

2026-08-25 15:17:22,342 | INFO | joint_test_metrics={'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 26462.0, 'fp': 59265.0, 'fn': 138.0, 'tp': 2187.0, 'accuracy': 0.32536455730704583, 'precision': 0.035588752196836555, 'recall': 0.9406451612903226, 'false_call_reduction': 0.3086775461639857, 'f1': 0.06858271790770967, 'roc_auc': 0.8714556448320381, 'pr_auc': 0.31860498449750563}


2026-08-25 15:17:22,343 | INFO | source_integrity=PASS


2026-08-25 15:17:22,343 | INFO | experiment_complete=0825_peace_014_joint_type_threshold_optimization
